____
### 1. Imports

In [1]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

____
### 2. Environment

Creating a custom environment to simulate the following condtions:
1. 30 day period
2. Required to sell inventory of 100 units
3. Unit cost of each unit set at $5
4. Agents in the environment are required to maximise profit in the 30 day period


Methods (1 - 3 are required for all environments):
1. `__init__()` → setup
2. `reset()` → start a new episode
3. `step(action)` → apply an action
4. `demand(price)` → calculates the demand given a price

Attributes required for all environments:
1. observation/action spaces


In [ ]:
class DynamicPricingEnv(gym.Env):
    def __init__(self):
        self.max_steps = 30
        self.max_inventory = 100
        self.cost = 5.0
        self.observation_space = gym.Box( #observation is given by [inventory, days left, number of units sold last]
            low=np.array([0, 0, 0]),
            high=np.array([self.max_inventory, self.max_steps, self.max_inventory]),
            dtype=np.float32
        )
        self.action_space = gym.Box(low=5.0, high=50.0, shape=(1,), dtype=np.float32) 
        #action is a discrete number between 5 to 50, stored as a 1D array 

    def reset(self, seed=None, options=None): #resets the values in the state
        self.inventory = self.max_inventory 
        self.step_count = 0
        self.last_demand = 0
        return self._obs(), {} #extra info dictionary is empty

    def step(self, action):
        price = float(action[0]) #obtainthe price from the action (indexing a 1d array)
        demand = self.demand(price) #calculating demand from the price set
        units_sold = min(demand, self.inventory) #

        self.inventory -= units_sold #updated inventory
        self.step_count += 1 # step + 1
        self.last_demand = units_sold # updating last known demand

        reward = (price - self.cost) * units_sold #reward is the profit generated from that round
        terminated = self.inventory <= 0
        truncated = self.step_count >= self.max_steps

        return self.obs(), reward, terminated, truncated, {}

    def obs(self): #to obtain observation of the current state
        return np.array([self.inventory, self.max_steps - self.step_count,
                         self.last_demand], dtype=np.float32)

    def demand(self, price):
        base = 40
        sensitivity = 1.2
        noise = np.random.normal(0, 3) #noise is random number from 0 to 3
        return int(max(0, base - sensitivity * price + noise))

____
#### 2.1 Exploring the environment